# AnchorDepth — Inference Demo

**Bachelor thesis demo** — Politehnica University of Timișoara, 2026

This notebook runs three monocular depth estimation models on a single user-uploaded image and compares their predictions and inference times side-by-side:

1. **AnchorDepth (ours)** — consistency-anchored adaptation of Depth Pro
2. **Depth Pro (zero-shot)** — Apple's foundation model, no KITTI training
3. **VGGT** — Visual Geometry Grounded Transformer (CVPR 2025)

Run cells top-to-bottom. The last cell lets you upload any RGB image and produces a side-by-side comparison figure with per-model inference times.

## 1. Install dependencies
Runs once per Colab session (~3 minutes the first time).

In [ ]:
!pip install -q huggingface_hub torch torchvision Pillow matplotlib
!pip install -q git+https://github.com/apple/ml-depth-pro.git
!pip install -q git+https://github.com/facebookresearch/vggt.git
print('✓ Dependencies installed')

## 2. Load all three models
Total download: ~5 GB.   Takes ~2 minutes the first time, cached afterwards.

In [ ]:
import torch, time
from huggingface_hub import hf_hub_download
import depth_pro

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# --- AnchorDepth (ours) -------------------------------------
print('Downloading AnchorDepth from Hugging Face...')
anchor_ckpt = hf_hub_download(repo_id='dariusan3/AnchorDepth', filename='anchordepth.pt')
model_anchor, _ = depth_pro.create_model_and_transforms(device=device)
model_anchor.load_state_dict(torch.load(anchor_ckpt, map_location=device), strict=True)
model_anchor.eval()
print('✓ AnchorDepth loaded')

# --- Depth Pro zero-shot -----------------------------------
model_zs, _ = depth_pro.create_model_and_transforms(device=device)
model_zs.eval()
print('✓ Depth Pro zero-shot loaded')

# --- VGGT ---------------------------------------------------
from vggt.models.vggt import VGGT
model_vggt = VGGT.from_pretrained('facebook/VGGT-1B').to(device)
model_vggt.eval()
print('✓ VGGT loaded')

## 3. Upload an image
Click the **Choose Files** button below and select any RGB image (JPG, PNG).

In [ ]:
from google.colab import files
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print(f'Image uploaded: {image_path}')

## 4. Run inference on all three models + display side-by-side comparison

In [ ]:
import numpy as np
from PIL import Image
from torchvision.transforms import Normalize, ToTensor
import matplotlib.pyplot as plt

img = Image.open(image_path).convert('RGB')
img_1536 = img.resize((1536, 1536), Image.LANCZOS)
norm = Normalize([0.5]*3, [0.5]*3)
inp = norm(ToTensor()(img_1536)).unsqueeze(0).to(device)

def predict_depthpro(model):
    """AnchorDepth / Depth Pro inference with timing."""
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad(), torch.amp.autocast(device.type):
        canon, fov = model(inp)
        f_px = 0.5 * 1536 / torch.tan(0.5 * torch.deg2rad(fov.float()))
        depth = 1.0 / torch.clamp(canon * (1536 / f_px), 1e-4, 1e4)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = (time.time() - t0) * 1000
    return 1.0 / depth.squeeze().cpu().float().numpy(), elapsed   # return inverse depth (better viz)

def predict_vggt(model):
    """VGGT inference with timing (single image, depth output only)."""
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    from vggt.utils.load_fn import load_and_preprocess_images
    imgs = load_and_preprocess_images([image_path]).to(device)
    if imgs.dim() == 4:
        imgs = imgs.unsqueeze(0)
    with torch.no_grad(), torch.amp.autocast(device.type, dtype=torch.bfloat16):
        preds = model(imgs)
    depth = preds['depth'][0, 0].squeeze().float().cpu().numpy() if 'depth' in preds else None
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = (time.time() - t0) * 1000
    return depth, elapsed

# Warm-up (first inference is slower due to CUDA kernel compilation)
_ = predict_depthpro(model_anchor)

# Run all three
print('Running inference...')
d_anchor, t_anchor = predict_depthpro(model_anchor)
d_zs,     t_zs     = predict_depthpro(model_zs)
d_vggt,   t_vggt   = predict_vggt(model_vggt)

print(f'AnchorDepth (ours):   {t_anchor:.1f} ms')
print(f'Depth Pro zero-shot:  {t_zs:.1f} ms')
print(f'VGGT:                 {t_vggt:.1f} ms')

# Plot side-by-side
def show(ax, d, title, ms):
    if d is None:
        ax.text(0.5, 0.5, 'no output', ha='center'); ax.axis('off'); return
    vmin, vmax = np.percentile(d[np.isfinite(d)], [5, 95])
    ax.imshow(np.clip(d, vmin, vmax), cmap='turbo')
    ax.set_title(f'{title}\n({ms:.0f} ms)', fontsize=13, fontweight='bold')
    ax.axis('off')

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
axes[0].imshow(img); axes[0].set_title('Input image', fontsize=13, fontweight='bold'); axes[0].axis('off')
show(axes[1], d_anchor, 'AnchorDepth ★ (ours)', t_anchor)
show(axes[2], d_zs,     'Depth Pro zero-shot',  t_zs)
show(axes[3], d_vggt,   'VGGT',                 t_vggt)
plt.tight_layout()
plt.savefig('comparison.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("\n✓ Saved comparison.png")

## 5. (optional) Download the figure to your computer

In [ ]:
from google.colab import files
files.download('comparison.png')